Full disclaimer,,,
 
I ripped tf outta this MIT lecture [here](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/pages/linear-regression/moneyball-the-power-of-sports-analytics/) where they did it in R.  

But I made it cooler and better. 

# ISBE 
* I - Inspect
* S - Select
* B - Build
* E - Evaluate 

---
# ⚾ The Story: Moneyball (2002 Oakland A's)

**The problem.** It's the end of the 2001 season. The Oakland Athletics just lost their three best players (Jason Giambi, Johnny Damon, and closer Jason Isringhausen) to teams that could pay them more. The A's have one of the **smallest payrolls in baseball, around \$40M**. The New York Yankees spend roughly **three times that**.

Rich teams can buy the best players. Poor teams can't. So in theory the rich teams should win every year.

**The idea.** A's General Manager **Billy Beane** and his analyst **Paul DePodesta** chose a different approach. Instead of trusting scouts' gut feelings ("he *looks* like a ballplayer"), they used **data** to ask:

> *What actually wins baseball games, and which of those skills are cheap to buy because everyone else undervalues them?*

This approach is called **sabermetrics**, a term popularised by the baseball writer Bill James.

**The plan we'll follow in this notebook.** DePodesta reduced the whole season to a chain of questions. Each one can be answered with a regression model:

```
 Make the playoffs  ←  Win ~95 games  ←  Big enough run differential (RS − RA)
                                               ↑                    ↑
                                       Runs Scored (RS)      Runs Allowed (RA)
                                         ↑ OBP, SLG?          ↑ OOBP, OSLG?
```

1. **How many wins do we need** to make the playoffs?
2. **How big a run differential** (runs scored − runs allowed) do we need to get those wins?
3. **Which hitting stats** drive *runs scored*? (Is it batting average, like the scouts believe?)
4. **Which pitching/defence stats** drive *runs allowed*?
5. Use those models to **predict the 2002 season** and to **find undervalued players**.

**Our time machine.** To be fair to Billy Beane, we can only use information he had *before* the 2002 season. That's why the data pipeline below **throws away every season from 2002 onward**.

We'll use the **ISBE** workflow at every step: **Inspect** the data, **Select** variables, **Build** a model, **Evaluate** it.

## 🧰 Step 0: Load our tools
Run this cell to import everything we need:
* `pandas` / `numpy` handle data tables and maths.
* `seaborn` / `matplotlib` draw plots.
* `statsmodels` is our main regression tool. Its `.summary()` report shows coefficients **and** p-values, which is what an analyst like DePodesta wants.
* `sklearn` is used for train/test splitting and scoring.

In [ ]:
# PANDAS IS FOR DATA WRANGLING
import pandas as pd
import numpy as np

# SEABORN IS A PLOTTING LIBRARY
import seaborn as sns

# MATPLOT LIB IS ALSO A PLOTTING LIBRARY
import matplotlib.pyplot as plt

# SKLEARN IS OUR MACHINE LEARNING PACKAGE
from sklearn.linear_model import LinearRegression

# IMPORT OUR RANDOM FOREST REGERSSOR
from sklearn.ensemble import RandomForestRegressor

# METRICS HELP US SCORE OUR MODEL
from sklearn import metrics

# HELP US SPLIT OUR DATA INTO TESTING A TRAINING
from sklearn.model_selection import train_test_split

# Good ol statsmodels
import statsmodels.api as sm

# Specific root mean squared error for stats models
from statsmodels.tools.eval_measures import rmse
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import qqplot

## 🔎 I: Inspect. Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
3. Creates **run differential**: `RS − RA` (runs scored minus runs allowed). It's stored as `runs_diff`.

> 💡 **Why run differential?** You win a baseball game by scoring more runs than the other team. Over a full 162-game season, a team that scores many more runs than it gives up should win a lot of games. Keep that idea in mind; we'll test it later.

In [ ]:
df = pd.read_csv('data/baseball.csv')

# Time machine: keep only seasons before Moneyball (2002)
df = df[df['Year'] < 2002].reset_index(drop=True)

# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['RS'] - df['RA']

df.head()

#### Data Dictionary

| **Column**       | **Description**                                          |
| ---------------- | -------------------------------------------------------- |
| **Team**         | Name of the team                                         |
| **League**       | Name of the league                                       |
| **Year**         | Year of the season                                       |
| **RS**           | Runs scored by the team                                  |
| **RA**           | Runs allowed by the team                                 |
| **W**            | Number of wins                                           |
| **OBP**          | On-Base Percentage                                       |
| **SLG**          | Slugging Percentage                                      |
| **BA**           | Batting Average                                          |
| **Playoffs**     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **RankSeason**   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **RankPlayoffs** | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **G**            | Number of games played                                   |
| **OOBP**         | Opponent On-Base Percentage                              |
| **OSLG**         | Opponent Slugging Percentage                             |
| **runs_diff**    | Runs scored minus runs allowed (`RS - RA`)               |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

### Sanity-check the data
Before modelling, always check three things: **shape**, **nulls**, and **duplicates**.

`df.info()` covers shape and nulls in one command: the row count, plus the **non-null count** for each column (a column with fewer non-null values than rows has missing data). Duplicates get their own check.

👀 **Look for:** which columns have lots of missing values? `RankSeason` and `RankPlayoffs` are only filled in for playoff teams, so the blanks make sense. **`OOBP` and `OSLG`** are missing for most of the older seasons. Remember this; we'll need to drop those rows when we model runs allowed.

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

---
# 🏏 Part 1: How do we SCORE runs?
The A's can't afford the "best" hitters, so they need to know **exactly which hitting skills produce runs**. Then they can shop for those skills at a discount.

Our **dependent variable** (what we're predicting) is `RS`, runs scored in a season.

# SCORING RUNS
_Lets build a Linear Regression model to see what is most important to predict how many runs we score from the data we have_

In [ ]:
df.head()

### S: Select (first attempt; spot the mistake!)
For our first try, we'll throw in **almost every numeric column** we have and see what happens. As you run the next few cells, think like a detective: *does it make sense to use these columns to predict runs scored?*

In [ ]:
dependent_variable = 'RS'
independent_variables = ['Year', 'OBP', 'SLG', 'BA', 'RA', 'W', 'Playoffs', 'G', 'runs_diff']

X = df[independent_variables]
y = df[dependent_variable]

X.head()

### B: Build
`statsmodels` needs us to add a **constant** (a column of 1s) so the model can learn an intercept. Then `.fit()` finds the best-fitting coefficients.

In [ ]:
import statsmodels.api as sm

## we have to add a constant to our X data (a column of just all 1's)
X = sm.add_constant(X)

# build our model
model_wins = sm.OLS(y, X).fit()

# make new predictions
y_pred = model_wins.predict(X)

### E: Evaluate
Read the summary report. Look at **R²**, the **coefficients**, and the **p-values** (`P>|t|`). Also compare **RMSE** with the average number of runs: RMSE tells you, on average, how many runs our predictions miss by.

In [ ]:
# get the summary report
results = model_wins.summary2()
print(results)


# get root mean squared error
model_wins_root_mean_squared_error = rmse(y, y_pred)
print('#'*79)
print( 'Dependent Variable %s average: ' % dependent_variable, y.mean())
print( 'RMSE:', model_wins_root_mean_squared_error)


# What is wonky about the Regression Summary above....?
0. What are we trying to predict...?
0. What are we using to predict that? 
0. What are the assumptions of Linear Regression. 

<details>
<summary><b>🧑‍🏫 Click for the answer (try first!)</b></summary>

* **We're cheating (data leakage).** `runs_diff` is literally `RS − RA`. If you know `runs_diff` and `RA`, you can calculate `RS` exactly, with no baseball knowledge needed. The model "predicts" perfectly because we handed it the answer.
* **Wrong direction of cause.** `W` (wins) and `Playoffs` are *results* of scoring runs, not causes. Billy Beane can't go out and buy "wins" to make his team score more runs.
* **Not useful for decisions.** `Year` and `G` (games played) aren't skills you can buy from a player.
* **Assumption check:** linear regression assumes the predictors aren't near-copies of each other (no **multicollinearity**). `RS`, `RA`, `runs_diff`, and `W` are tangled together.

**Rule of thumb:** only use predictors that (1) you would **know before the outcome happens**, and (2) you can actually **act on**. For a GM, that means player skills like OBP, SLG, and BA.
</details>

### 📊 Visualise each variable against runs scored
A picture often shows the problem faster than a summary table. Each mini-plot shows one variable against `RS`, with a red best-fit line.

👀 **Look for:** which plots show a sensible, real-world relationship between a **player skill** and runs (`OBP`, `SLG`, `BA`)?

Notice that `runs_diff` and `RA` each look only loosely related to `RS` on their own. The leakage comes from using them **together**: `RS = runs_diff + RA` exactly. No single scatter plot can show that, which is why the regression looked perfect even though none of these plots do.

In [ ]:
sns.pairplot(df, x_vars=['Year', 'OBP', 'SLG'], y_vars=['RS'], kind='reg', height=3,
             plot_kws={'scatter_kws': {'s': 5, 'alpha': 0.4}, 'line_kws': {'color': 'red'}})
plt.show()

In [ ]:
sns.pairplot(df, x_vars=['BA', 'RA', 'W'], y_vars=['RS'], kind='reg', height=3,
             plot_kws={'scatter_kws': {'s': 5, 'alpha': 0.4}, 'line_kws': {'color': 'red'}})
plt.show()

In [ ]:
sns.pairplot(df, x_vars=['Playoffs', 'G', 'runs_diff'], y_vars=['RS'], kind='reg', height=3,
             plot_kws={'scatter_kws': {'s': 5, 'alpha': 0.4}, 'line_kws': {'color': 'red'}})
plt.show()

#### Okay lets fix it with proper signals aka variables 

We keep only the three **hitting skills** a GM could actually shop for: **OBP, SLG, and BA**. This is the exact question the A's front office faced: *which of these three should we pay for?*

In [ ]:
dependent_variable = 'RS'
independent_variables = ['OBP', 'SLG', 'BA']

X = df[independent_variables]
y = df[dependent_variable]


## we have to add a constant to our X data (a column of just all 1's)
X = sm.add_constant(X)

# build our model
model_wins = sm.OLS(y, X).fit()

# make new predictions
y_pred = model_wins.predict(X)

# get the summary report
results = model_wins.summary2()
print(results)


# get root mean squared error
model_wins_root_mean_squared_error = rmse(y, y_pred)
print('#'*79)
print( 'Dependent Variable %s average: ' % dependent_variable, y.mean())
print( 'RMSE:', model_wins_root_mean_squared_error)


# Okay, with the summary report above for OBP, SLG, and BA. 
1. Our r^2 score looks great (0.93) 
2. However, look closely at the coefficients. 
3. Batting Average (which is a good thing) is negative,,, meaning the higher the batting average is the less runs you score. 
4. That does not make sense at all.... 


## Why is this happening.  

In [ ]:
df[independent_variables].corr().round(4)

👀 **Look at the correlation table.** All three stats are strongly correlated with each other (0.8 or higher). That makes sense: a player who gets lots of hits usually also gets on base a lot and hits for power.

This is called **multicollinearity**. When predictors overlap this much, the model can't tell which one deserves the credit. It splits the credit oddly, and a coefficient can even **flip sign**. That's why BA came out *negative*. It doesn't mean hits hurt you; it means BA has almost nothing extra to add once OBP and SLG are already in the model.

<div>
<img src=https://i.imgflip.com/80a3lx.jpg width="500">
</div>

# We are going to do a lot of regression models. So lets put it into a resuable function.

### Two helper functions
* **`eval_regression`** splits the data into **80% training / 20% testing**, fits on the training part, and scores on the testing part. This checks whether the model works on seasons it has **never seen**, which is the real test for predicting 2002.
* **`do_moneyball`** fits the model on **all** the data, prints R² and RMSE, calls `eval_regression` for the held-out score, and (optionally) prints the full summary. It **returns the fitted model** so we can use it to make predictions later.

> 📝 Note: the first block printed by `do_moneyball` is labelled "EVAL ON TESTING DATA", but it's really the score on the **full** dataset. The second block (from `eval_regression`) is the true held-out test score.

In [ ]:
def eval_regression(input_df, independent_variables, dependent_variable):
    data = input_df.copy()

    # Define the independent and dependent variables
    X = data[independent_variables]  # Independent variable
    y = data[dependent_variable]  # Dependent variable


    # Add a constant to the independent variables (required by statsmodels)
    X = sm.add_constant(X)

    # Split the data into training and test sets (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Build the linear regression model
    model = sm.OLS(y_train, X_train).fit()

    y_pred = model.predict(X_test)

    # Rounding the massive long float
    y_pred = round(y_pred,4)

    # Calculate R^2
    r_squared = metrics.r2_score(y_test, y_pred)

    # Rounding the massive long float
    r_squared = round(r_squared, 4)

    # Calculate RMSE
    model_root_mean_squared_error = rmse(y_test, y_pred)

    # Rounding the massive long float
    model_root_mean_squared_error = round(model_root_mean_squared_error, 4)

    print('R-Squared Score:', r_squared)
    print('RMSE:', model_root_mean_squared_error)
    print( 'Dependent Variable %s average: ' % dependent_variable, round(y.mean(), 4))
    precentage_off = round(model_root_mean_squared_error / y.mean() , 4)
    print(f'On average our prediction is %{precentage_off} the dependent variable')

    return None

In [ ]:
def do_moneyball(input_df, independent_variables, dependent_variable, print_model_summary=True):

    # Copy the data
    data = input_df.copy()

    # Define the independent and dependent variables
    X = data[independent_variables]  # Independent variable
    y = data[dependent_variable]  # Dependent variable

    # Add a constant to the independent variables (required by statsmodels)
    X = sm.add_constant(X)

    # Build the linear regression model
    model = sm.OLS(y, X).fit()

    # Make predictions
    y_pred = model.predict(X)


    # Calculate R^2
    r_squared = metrics.r2_score(y, y_pred)
    r_squared = round(r_squared, 4)


    # Calculate RMSE
    model_root_mean_squared_error = rmse(y, y_pred)
    model_root_mean_squared_error = round(model_root_mean_squared_error, 4)

    # Print model description
    #print('\n'*2)
    print("#"*79)
    print("Predicting:", dependent_variable)
    print("Using:",  independent_variables)
    print("#"*79)
    #print('\n'*2)

    # Print eval metrics
    print('/'*79)
    print('EVAL ON TESTING DATA')
    print('R-Squared Score:', r_squared)
    print('RMSE:', model_root_mean_squared_error)
    print('Dependent Variable %s average: ' % dependent_variable, round(y.mean(), 3))
    print('/'*79)
    #print('\n'*2)

    print('-'*79)
    print('EVAL ON TESTING DATA')
    print(eval_regression(input_df, independent_variables, dependent_variable))
    print('-'*79)
    #print('\n'*2)
    # Return model

    # Print the summary of the model if you want it
    if print_model_summary:
        print(model.summary())

    return(model)



selected_features = ['OBP', 'SLG', 'BA']
target_feature = 'RS'

model = do_moneyball(df, independent_variables=selected_features, dependent_variable=target_feature)

# To mitigate the multicolinearity, lets try all the combinations of the variables. 

We'll try each **pair** of stats and compare. For each run, write down:
1. **R²**: how much of the variation in runs is explained (higher is better)
2. **RMSE**: the typical miss, in runs per season (lower is better)
3. Do the **coefficients make sense** (positive) and are the **p-values small**?

| Predictors | R² | RMSE | Coefficients make sense? |
|---|---|---|---|
| BA + SLG | | | |
| BA + OBP | | | |
| OBP + SLG | | | |

In [ ]:
X_vars = ['BA', 'SLG']
y_var = 'RS'
do_moneyball(df, X_vars, y_var, print_model_summary=True)

In [ ]:
X_vars = ['BA', 'OBP']
y_var = 'RS'
do_moneyball(df, X_vars, y_var)

In [ ]:
X_vars = ['OBP', 'SLG']
y_var = 'RS'
do_moneyball(df, X_vars, y_var)


<details>
<summary><b>🧑‍🏫 Click to compare with the instructor's results</b></summary>

| Predictors | R² (all data) | RMSE | Coefficients make sense? |
|---|---|---|---|
| BA + SLG | 0.876 | ~33 runs | ✅ |
| BA + OBP | 0.832 | ~38 runs | ✅ |
| **OBP + SLG** | **0.930** | **~25 runs** | ✅ |
| OBP + SLG + BA | 0.930 | ~25 runs | ❌ BA is negative |

**OBP + SLG** matches the three-variable model's accuracy *without* the nonsense negative coefficient. Adding BA bought us nothing. **Batting average, the scouts' favourite stat, is the one we can drop.** That's the heart of Moneyball.
</details>

In [ ]:
X_vars = ['OBP', 'SLG']
y_var = 'RS'
model_OBP_SLG = do_moneyball(df, X_vars, y_var, print_model_summary=True)


# Using SLG and OBP have the best r^2 score with significatn p-values 
## Thus we can conclude that OBP and SLG are the best signals for scoring runs. 

Holding slugging constant, a team that improves its OBP by 0.010 (for example, from .320 to .330) is expected to score about 27 more runs in a season.

```2737.77 × 0.01 = 27.38``` more runs per season.

### 💰 Why this matters for the A's
Look at the two coefficients: **OBP ≈ 2738** and **SLG ≈ 1585**. OBP and SLG are measured on roughly similar scales, so **a point of OBP is worth noticeably more runs than a point of SLG**.

The market, though, priced players by **batting average and power**. Players who drew lots of walks (high OBP, ordinary BA) were **cheap**. DePodesta's insight: *buy on-base percentage, because nobody else is paying for it.*

### 🧪 Try the model on a real team: the 2000 Oakland A's
Let's pull the A's row from the 2000 season and plug their **OBP and SLG** into our model. Then compare the prediction with how many runs they actually scored (`RS`).

In [ ]:
#so, using our model.  we can now see how many runs we should score with any BA and SLG...
# year 2000's stats, we can see how many runs we will earn
team_filter = df.Team == 'OAK'
year_filter = df.Year == 2000

tmp_df = df[(team_filter) & (year_filter)].copy()
# tmp_df = tmp_df[X_vars]
tmp_df

In [ ]:
model_OBP_SLG.predict([1, 0.36, 0.458])

👀 The model predicts about **907 runs** and the 2000 A's actually scored **947**. That's within about 40 runs, using only two numbers. Not bad!

> 📝 The `[1, 0.36, 0.458]` list is `[constant, OBP, SLG]`, in the same order as the columns we trained on. The leading `1` is for the intercept.

---
# 🧤 Part 2: How do we ALLOW fewer runs?
Winning is about **run differential**, so preventing runs counts just as much as scoring them. Let's flip the question around and use the **opponents'** hitting stats against our pitchers:

* **Dependent variable:** `RA` (runs allowed)
* **Try predictors:** `OOBP` (opponents' on-base %) and `OSLG` (opponents' slugging %)

⚠️ Remember the null check: `OOBP` and `OSLG` are **missing for most of the older seasons**. Drop those rows first (e.g. `df.dropna(subset=['OOBP', 'OSLG'])`), or the model will error out.

**Your turn:** use `do_moneyball` to build this model. Do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model?

In [ ]:
# LETS NOW DO THE SAME THING, BUT FOR FINDING WHICH VARIBALES LEAD TO RUNS ALLOWED....




In [ ]:
### LIVE CODE THE OOBP and OOBA

---
# 💵 Part 3: Finding undervalued players
We now know **OBP** is a big driver of runs. The next question is the money one: *which players have a high OBP but a low salary?* Those are the players the A's can afford.

This section switches to a **player-level** dataset: one row per player per season, with salaries.

# Show me the money!

In [ ]:
# Load the enhanced moneyball dataset
enhanced_df = pd.read_csv('data/enhanced_moneyball_analysis_1990_2010.csv')

# Print basic dataset information
print("=== ENHANCED MONEYBALL DATASET ANALYSIS ===")
print(f"Total number of records: {enhanced_df.shape[0]:,}")
print(f"Year range: {enhanced_df['yearID'].min()} - {enhanced_df['yearID'].max()}")
print(f"Number of unique players: {enhanced_df['playerID'].nunique():,}")

# Filter for pre-Moneyball era (yearID <= 2001) and (salary > 0)
pre_moneyball_df = enhanced_df[(enhanced_df['yearID'] <= 2001) & (enhanced_df['salary'] > 0)].copy()

# pre_moneyball_df = pre_moneyball_df[pre_moneyball_df.AB >= 55]

print(f"\n=== PRE-MONEYBALL ERA (≤2001) ANALYSIS ===")
print(f"Total number of records: {pre_moneyball_df.shape[0]:,}")
print(f"Year range: {pre_moneyball_df['yearID'].min()} - {pre_moneyball_df['yearID'].max()}")
print(f"Number of unique players: {pre_moneyball_df['playerID'].nunique():,}")

pre_moneyball_df.head()

Again we use our **time machine**: keep only seasons up to 2001 and players with a recorded salary. This is the "free-agent shopping list" DePodesta could have built.

### The quadrant chart
We split players into four groups using two cut-offs:
* **High OBP**: top 25% of on-base percentage
* **Low salary**: bottom 25% of pay

|  | Low salary | High salary |
|---|---|---|
| **High OBP** | 🟢 **Undervalued: buy these!** | ⚪ Good, but you pay for it |
| **Low OBP** | 🔵 Cheap and you get what you pay for | 🔴 **Overvalued: avoid** |

👀 **Look for:** how *few* players land in the green box compared with the red box. Hover over the dots to see who they are. Scroll the printed list for a familiar name: **Jeremy Giambi (2001 OAK)**, one of the real players in the Moneyball story.

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm
import warnings

from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor


# Remove rows with missing OBP or salary data for the analysis
analysis_df = pre_moneyball_df.dropna(subset=['OBP', 'salary']).copy()

# Calculate percentiles to identify undervalued players
# High OBP = above 75th percentile
# Low Salary = below 25th percentile
obp_75th = analysis_df['OBP'].quantile(0.75)
salary_25th = analysis_df['salary'].quantile(0.25)

print(f"=== QUADRANT ANALYSIS CRITERIA ===")
print(f"High OBP threshold (75th percentile): {obp_75th:.3f}")
print(f"Low Salary threshold (25th percentile): ${salary_25th:,.0f}")

# Create quadrant categories for each player
def assign_quadrant(row):
    if row['OBP'] >= obp_75th and row['salary'] <= salary_25th:
        return 'High OBP, Low Salary (Green)'  # Undervalued
    elif row['OBP'] < obp_75th and row['salary'] <= salary_25th:
        return 'Low OBP, Low Salary (Blue)'
    elif row['OBP'] >= obp_75th and row['salary'] > salary_25th:
        return 'High OBP, High Salary (Gray)'
    else:
        return 'Low OBP, High Salary (Red)'  # Overvalued

analysis_df['quadrant'] = analysis_df.apply(assign_quadrant, axis=1)

# Count players in each quadrant
quadrant_counts = analysis_df['quadrant'].value_counts()
print(f"\n=== QUADRANT DISTRIBUTION ===")
for quadrant, count in quadrant_counts.items():
    print(f"{quadrant}: {count} players")

# Identify undervalued players (green quadrant)
undervalued_players = analysis_df[analysis_df['quadrant'] == 'High OBP, Low Salary (Green)'].copy()
print(f"\nNumber of undervalued players identified: {len(undervalued_players)}")

# Create interactive scatter plot with quadrant-based coloring
fig = px.scatter(
    analysis_df,
    x='salary',
    y='OBP',
    color='quadrant',  # Color by quadrant
    size='G',  # Games played as size
    size_max=8,  # Smaller maximum bubble size
    hover_data=['playerID', 'teamID', 'yearID', 'BA', 'SLG', 'OPS'],
    title='Quadrant Analysis: OBP vs Salary (Pre-Moneyball Era)',
    labels={
        'salary': 'Salary ($)',
        'OBP': 'On-Base Percentage (OBP)',
        'G': 'Games Played',
        'quadrant': 'Quadrant'
    },
    color_discrete_map={
        'High OBP, Low Salary (Green)': 'green',
        'Low OBP, Low Salary (Blue)': 'blue',
        'High OBP, High Salary (Gray)': 'gray',
        'Low OBP, High Salary (Red)': 'red'
    }
)

# Add threshold lines
fig.add_hline(y=obp_75th, line_dash="dash", line_color="red",
              annotation_text=f"High OBP Threshold ({obp_75th:.3f})")
fig.add_vline(x=salary_25th, line_dash="dash", line_color="red",
              annotation_text=f"Low Salary Threshold (${salary_25th:,.0f})")

# Quadrant-based coloring is now handled by the main scatter plot

# Update layout
fig.update_layout(
    width=1200,
    height=600,
    showlegend=False,
    xaxis_title="Salary ($)",
    yaxis_title="On-Base Percentage (OBP)",
    title_x=0.5
)

# Add quadrant labels
fig.add_annotation(
    x=0.95, y=0.95,
    xref="paper", yref="paper",
    text="<b>High OBP<br>High Salary</b>",
    showarrow=False,
    font=dict(size=12, color="gray"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
)

fig.add_annotation(
    x=0.05, y=0.95,
    xref="paper", yref="paper",
    text="<b>High OBP<br>Low Salary</b><br><span style='color:green'>UNDERVALUED</span>",
    showarrow=False,
    font=dict(size=12, color="green"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="green",
    borderwidth=2
)

fig.add_annotation(
    x=0.95, y=0.05,
    xref="paper", yref="paper",
    text="<b>Low OBP<br>High Salary</b><br><span style='color:red'>OVERVALUED</span>",
    showarrow=False,
    font=dict(size=12, color="red"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="red",
    borderwidth=1
)

fig.add_annotation(
    x=0.05, y=0.05,
    xref="paper", yref="paper",
    text="<b>Low OBP<br>Low Salary</b>",
    showarrow=False,
    font=dict(size=12, color="blue"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="blue",
    borderwidth=1
)

fig.show()

# Display top undervalued players if any exist
if len(undervalued_players) > 0:
    print(f"\n=== UNDERVALUED PLAYERS (High OBP (descending), Low Salary) ===\n")
    top_undervalued = undervalued_players.sort_values(['OBP'], ascending=[False])[['playerID', 'yearID', 'teamID', 'OBP', 'BA', 'SLG', 'OPS', 'salary']]
    print(top_undervalued.to_string(index=False))
else:
    print("\nNo players meet the strict criteria for undervalued players.")
    print("Let's look at players with above-average OBP and below-average salary...")

    # Let us relax the criteria to find more undervalued players.
    # Relaxed criteria:
    # 1. Above median OBP
    # 2. Below median salary
    obp_median = analysis_df['OBP'].median()
    salary_median = analysis_df['salary'].median()

    relaxed_undervalued = analysis_df[
        (analysis_df['OBP'] >= obp_median) &
        (analysis_df['salary'] <= salary_median)
    ].copy()

    print(f"\nRelaxed criteria (above median OBP: {obp_median:.3f}, below median salary: ${salary_median:,.0f})")
    print(f"Number of players: {len(relaxed_undervalued)}")

    if len(relaxed_undervalued) > 0:
        top_relaxed = relaxed_undervalued.sort_values(['OBP'], ascending=[False])[['playerID', 'yearID', 'teamID', 'OBP', 'BA', 'SLG', 'OPS', 'salary']]
        print(top_relaxed.to_string(index=False))

---
# 🏆 Part 4: How many wins does it take to make the playoffs?
Now we zoom out to the big goal. DePodesta looked at past seasons and estimated that **about 95 wins** almost guarantees a playoff spot.

👀 In the plot below, each dot is a team-season, coloured by whether it made the playoffs. The red line is at **95 wins**. Notice that **nearly every team to the right of the line made the playoffs**, and teams to the left are hit or miss.

In [ ]:
sns.set()
plt.figure(figsize=(13, 8))
ax = sns.scatterplot(df, x='W', y='Team', hue='Playoffs')
ax.axvline(x=95, color='red')


# HOW TO WIN 95 GAMES...

### Step 1: Turn "95 wins" into a run-differential target
A GM can't buy "wins" directly, but he *can* build a roster that scores more runs than it allows. So let's model **wins as a function of run differential**:

$$\text{Wins} = b_0 + b_1 \times \text{RunDiff}$$

👀 **Look for:** R² is about 0.88, so run differential explains most of the difference in wins between teams. Notice the intercept is **≈ 81 wins**. A team that scores exactly as many runs as it allows wins about half its 162 games. That's a nice sanity check!

In [ ]:
what_we_are_trying_to_predict = 'W'
data_we_have_related_to_games = ['runs_diff']
model = do_moneyball(df, data_we_have_related_to_games, what_we_are_trying_to_predict)

# We use this model to find our target runs difference we need to get 95 wins. 

Set wins = 95 and solve the line for run differential:

$$95 = 80.88 + 0.1058 \times \text{runs\_diff} \quad\Rightarrow\quad \text{runs\_diff} = \frac{95 - 80.88}{0.1058} \approx 133$$

**The A's need to outscore their opponents by roughly 133–135 runs over the season.** DePodesta reached essentially the same number. Now the problem is concrete: build a roster that scores about 135 more runs than it allows.

> 🧮 The rough cell below uses rounded numbers (80 and 0.1) to show the algebra by hand. The cell after it uses the model's real coefficients.

In [ ]:
# Using these coefficients...  find 95


# 95 >= 80 + 0.1*x

# x is what we are hunting for.

########################################
# 95-80 = 0.1x

# 15 = 0.1x

# 15/0.1 = 150

In [ ]:
coef_y_int = model.params['const']
coef_runs_diff = model.params['runs_diff']

## y = m*x + b
## x = y-b / m

### THIS IS SAME AS ABOVE
y_intercept = 80.88
m_coefficient = 0.1058
# x = ???

y = 95
x_target = (y - y_intercept) / m_coefficient
x_target


---
# 🔮 Part 5: Predict the 2002 season
Here's the full chain in action. It's the 2001–02 off-season. We know the stats of the players on the 2002 roster from their **2001** seasons, so we can:

1. Predict **runs scored** from the team's OBP and SLG
2. Predict **runs allowed** from opponents' OOBP and OSLG
3. Subtract to get **predicted run differential**
4. Feed that into the **wins** model

First we rebuild the two run models (scored and allowed) using all pre-2002 data.

In [ ]:
independent_variables = ['OBP', 'SLG']
dependent_variable = 'RS'

X = df[independent_variables]
y = df[dependent_variable]

X = sm.add_constant(X)

model_runs_scored = sm.OLS(y, X).fit()
#predictions = mode_runs_scored.predict(X)
print(model_runs_scored.summary())

In [ ]:
dfo = df.dropna(subset=['OOBP', 'OSLG'])
independent_variables = ['OOBP', 'OSLG']
dependent_variable = 'RA'

X = dfo[independent_variables]
y = dfo[dependent_variable]

X = sm.add_constant(X)

model_runs_allowed = sm.OLS(y, X).fit()
#predictions = mode_runs_scored.predict(X)
print(model_runs_allowed.summary())

### Plug in the 2002 A's (estimated from their players' 2001 stats)
* Team **OBP = .339**, **SLG = .430**
* Opponents **OOBP = .307**, **OSLG = .373**

👀 Is the predicted run differential above our **≈133-run target**?

In [ ]:
### FUTURE 2001 A's would have

const = 1
OBP = 0.339
SLG = 0.430

predicted_runs_scored = model_runs_scored.predict([const, OBP, SLG])
print(predicted_runs_scored)


## FUTURE OPPONENTS WOULD HAVE
OOBP = 0.307
OSLG = 0.373
predicted_runs_allowed = model_runs_allowed.predict([const, OOBP, OSLG])
print(predicted_runs_allowed)

predicted_runs_diff = predicted_runs_scored - predicted_runs_allowed
predicted_runs_diff

In [ ]:
independent_variables = ['runs_diff',]
dependent_variable = 'W'

X = df[independent_variables]
y = df[dependent_variable]

X = sm.add_constant(X)

model_wins = sm.OLS(y, X).fit()
#predictions = mode_runs_scored.predict(X)
# print(model_wins.summary())
model_wins.predict([1, predicted_runs_diff[0]])


### 🎉 How did the model do?

| | **Our model's prediction** | **What actually happened in 2002** |
|---|---|---|
| Runs scored | ~805 | 800 |
| Runs allowed | ~622 | 653 |
| Wins | ~100 | **103** |

Using a few simple linear regressions and data from **before the season started**, we predicted the A's would win about 100 games. They won **103**, made the playoffs, and set an American League record with a **20-game winning streak**, all on one of the smallest payrolls in baseball.

> 🧠 **Takeaway for data scientists:** the models weren't fancy. The win came from asking the **right question**, choosing **sensible variables**, and **trusting the evidence** over conventional wisdom.

> ⚾ **Epilogue:** the A's made the playoffs in 2002 but lost in the first round. Beane famously argued that the playoffs are largely luck: a short series doesn't give skill enough games to show itself. Statistically, that's the **sample size** problem: 162 games tell you a lot; 5 games tell you very little.